# Eksperimen 5: The Master Pipeline (Max Effort Tuning)
Mahakarya notebook kompetisi Kaggle untuk prediksi Tinggi Muka Air (TMA).
Tujuan utama notebook ini adalah menganalisis dataset secara komprehensif, mengeksploitasi relasi fisika lingkungan, dan memadukan kekuatan *Triple-Ensemble* (LightGBM, XGBoost, CatBoost) menggunakan **K-Fold Time-Series Cross Validation** serta **Optuna Hyperparameter Tuning** berskala masif.

Proses eksekusi *notebook* ini dirancang untuk memprioritaskan skor absolut, mengesampingkan efisiensi waktu komputasi.

In [1]:
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from sklearn.model_selection import TimeSeriesSplit
import optuna
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Integrasi & Pemuatan Data
Memuat keseluruhan data mentah (Train, Test, Data Lingkungan, dan Koordinat). Data *test* dikondisikan targetnya bernilai kosong.

In [2]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
env_data = pd.read_csv('../data/raw/data_pendukung/data_lingkungan.csv')
coords = pd.read_csv('../data/raw/data_pendukung/koordinat_pos.csv')

train['datetime'] = pd.to_datetime(train['datetime'])
test['datetime'] = pd.to_datetime(test['id'].str[:19])
test['nama_pos'] = test['id'].str[22:]
env_data['datetime'] = pd.to_datetime(env_data['datetime'])

## 2. Exploratory Data Analysis (EDA)
Inspeksi fundamental untuk mengetahui struktur matriks, mengkalkulasi anomali nilai kosong, dan mengecek integritas rentang waktu data.

In [3]:
print("=== Dimensi Matriks ===")
print("Train:", train.shape)
print("Test:", test.shape)
print("Lingkungan:", env_data.shape)

print("\n=== Cek Kekosongan Data (Missing Values) ===")
print("Anomali Data Lingkungan:")
print(env_data.isnull().sum()[env_data.isnull().sum() > 0])

=== Dimensi Matriks ===
Train: (84396, 3)
Test: (21780, 3)
Lingkungan: (888480, 27)

=== Cek Kekosongan Data (Missing Values) ===
Anomali Data Lingkungan:
soil_moisture_0_7cm          720
soil_moisture_7_28cm         720
soil_moisture_28_100cm       720
soil_moisture_100_255cm      720
surface_pressure_hpa         720
pressure_msl_hpa             720
rmm1                         720
rmm2                         720
mjo_phase                    720
mjo_amplitude                720
mjo_active                   720
nino_34                    12960
dtype: int64


### 2.1 Pengecekan Kontinuitas Kronologis
Memverifikasi batas transisi waktu perpindahan dari masa pelatihan ke masa ujian (*test set*).

In [4]:
print("Transisi Akhir Train:", train['datetime'].max())
print("Transisi Awal Test  :", test['datetime'].min())
print("Jeda Waktu          :", test['datetime'].min() - train['datetime'].max())

Transisi Akhir Train: 2025-09-18 18:00:00
Transisi Awal Test  : 2025-09-19 06:00:00
Jeda Waktu          : 0 days 12:00:00


### 2.2 Confirmatory EDA (CEDA) - Matriks Korelasi Lingkungan
Menarik koefisien korelasi linear antar berbagai parameter atmosfer dan tanah terhadap peninggian debit air (*tma_mdpl*).

In [5]:
train_temp = pd.merge(train, env_data, on=['datetime', 'nama_pos'], how='left')
num_cols = train_temp.select_dtypes(include=[np.number]).columns

corr = train_temp[num_cols].corr()['tma_mdpl'].sort_values(ascending=False)
print("=== Top Relasi Korelasi TMA ===")
print(corr.head(5))
print("...")
print(corr.tail(5))

=== Top Relasi Korelasi TMA ===
tma_mdpl                   1.000000
soil_moisture_100_255cm    0.189417
built_surface_m2           0.179301
soil_moisture_28_100cm     0.126973
soil_moisture_7_28cm       0.123752
Name: tma_mdpl, dtype: float64
...
rainfall_max_24h_mm    -0.025054
temperature_c          -0.076961
dew_point_c            -0.121663
landcover_class        -0.157293
surface_pressure_hpa   -0.947417
Name: tma_mdpl, dtype: float64


## 3. Imputasi Adaptif & Pengklasteran Geospasial
Parameter cuaca bergradasi halus seperti tekanan atmosfer diinterpolasi secara linear. Variabel iklim makro lambat seperti ENSO dikunci dengan substitusi ke depan.

In [6]:
env_data = env_data.sort_values(['nama_pos', 'datetime'])

macro_cols = ['nino_34', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'rmm1', 'rmm2']
dynamic_cols = ['surface_pressure_hpa', 'pressure_msl_hpa', 'soil_moisture_0_7cm', 
                'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm']

for c in macro_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].ffill().bfill()
    
for c in dynamic_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].apply(lambda x: x.interpolate(method='linear').bfill().ffill()).reset_index(level=0, drop=True)

kmeans = KMeans(n_clusters=5, random_state=42)
coords['spatial_cluster'] = kmeans.fit_predict(coords[['latitude', 'longitude']])

### 3.1 Penyelarasan Frekuensi Waktu (Agregasi)
Mentransformasikan sensor per-jam ke skala 3-jam untuk sinkronisasi mutlak dengan instrumen pantau TMA.

In [7]:
def aggregate_env_data(df):
    cat_cols = ['nama_pos', 'landcover_name', 'datetime']
    num_cols = [c for c in df.columns if c not in cat_cols]
    
    agg_funcs = {col: 'mean' for col in num_cols}
    agg_funcs['rainfall_mm'] = 'sum'
    agg_funcs['rainfall_openmeteo_mm'] = 'sum'
    agg_funcs['rainfall_max_24h_mm'] = 'max'
    
    df_indexed = df.set_index('datetime')
    agg_df = df_indexed.groupby(['nama_pos', pd.Grouper(freq='3h', label='right', closed='right')]).agg(agg_funcs).reset_index()
    return agg_df

env_agg = aggregate_env_data(env_data)

## 4. Deep Exogenous Engineering (Massive Lags)
Rekayasa interaksi hidrologi. Curah hujan diagregasikan secara ekstensif hingga mundur berhari-hari untuk mensimulasikan gelombang luapan jarak jauh.

In [8]:
test['tma_mdpl'] = np.nan
all_data = pd.concat([train, test], ignore_index=True)
all_data = all_data.sort_values(by=['nama_pos', 'datetime']).reset_index(drop=True)

all_data = pd.merge(all_data, env_agg, on=['datetime', 'nama_pos'], how='left')
all_data = pd.merge(all_data, coords, on='nama_pos', how='left')

all_data['month'] = all_data['datetime'].dt.month
all_data['hour'] = all_data['datetime'].dt.hour
all_data['sin_hour'] = np.sin(2 * np.pi * all_data['hour'] / 24)
all_data['cos_hour'] = np.cos(2 * np.pi * all_data['hour'] / 24)

all_data['runoff_factor'] = all_data['rainfall_mm'] * all_data['soil_moisture_0_7cm']
all_data['pressure_drop'] = all_data.groupby('nama_pos')['surface_pressure_hpa'].diff(1).fillna(0)

def create_massive_rolling(df):
    df_temp = df.copy()
    windows = [4, 8, 24, 56]
    
    for w in windows:
        df_temp[f'rainfall_roll_{w}'] = df_temp.groupby('nama_pos')['rainfall_mm'].transform(lambda x: x.rolling(window=w, min_periods=1).sum())
        df_temp[f'soil_moist_roll_{w}'] = df_temp.groupby('nama_pos')['soil_moisture_0_7cm'].transform(lambda x: x.rolling(window=w, min_periods=1).mean())
        
    return df_temp

all_data = create_massive_rolling(all_data)

le = LabelEncoder()
all_data['nama_pos_encoded'] = le.fit_transform(all_data['nama_pos'])

## 5. Setup K-Fold Time-Series Validation
Membangun perlindungan berlapis (5 Folds Walk-Forward) untuk mengeliminasi *overfitting*.

In [9]:
train_mask = all_data['tma_mdpl'].notnull()
train_data = all_data[train_mask].sort_values('datetime')
test_data = all_data[~train_mask].sort_values('datetime')

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'id', 'landcover_name']
features = [c for c in train_data.columns if c not in drop_cols]
target = 'tma_mdpl'

X_train_full = train_data[features].reset_index(drop=True)
y_train_full = train_data[target].reset_index(drop=True)
X_test_full = test_data[features].reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=5)

### 5.1 Max Effort: Optuna Tuning for LightGBM
Mencari konfigurasi paramter daun dan kecepatan belajar mutlak terbaik secara iteratif melintasi lipatan waktu.

In [10]:
def objective_lgb(trial):
    params = {
        'n_estimators': 300,
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'subsample': trial.suggest_uniform('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'verbose': -1
    }
    
    scores = []
    for train_index, val_index in tscv.split(X_train_full):
        X_tr, X_va = X_train_full.iloc[train_index], X_train_full.iloc[val_index]
        y_tr, y_va = y_train_full.iloc[train_index], y_train_full.iloc[val_index]
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        scores.append(mean_squared_error(y_va, preds))
        
    return np.mean(scores)

print("Memulai Optuna Tuning untuk LightGBM...")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=30)
best_lgb_params = study_lgb.best_params
best_lgb_params['n_estimators'] = 800
best_lgb_params['random_state'] = 42
best_lgb_params['verbose'] = -1
print("LightGBM Best Params:", best_lgb_params)

Memulai Optuna Tuning untuk LightGBM...
LightGBM Best Params: {'learning_rate': 0.018668631939045395, 'num_leaves': 255, 'max_depth': 5, 'subsample': 0.9948613944561775, 'colsample_bytree': 0.6478123523749714, 'n_estimators': 800, 'random_state': 42, 'verbose': -1}


### 5.2 Max Effort: Optuna Tuning for XGBoost
Algoritma sekunder dikerahkan untuk menemukan pohon terdalam dan proporsi baris yang paling tangguh.

In [11]:
def objective_xgb(trial):
    params = {
        'n_estimators': 300,
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_uniform('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }
    
    scores = []
    for train_index, val_index in tscv.split(X_train_full):
        X_tr, X_va = X_train_full.iloc[train_index], X_train_full.iloc[val_index]
        y_tr, y_va = y_train_full.iloc[train_index], y_train_full.iloc[val_index]
        
        model = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr, verbose=False)
        preds = model.predict(X_va)
        scores.append(mean_squared_error(y_va, preds))
        
    return np.mean(scores)

print("\nMemulai Optuna Tuning untuk XGBoost...")
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=30)
best_xgb_params = study_xgb.best_params
best_xgb_params['n_estimators'] = 800
best_xgb_params['random_state'] = 42
print("XGBoost Best Params:", best_xgb_params)


Memulai Optuna Tuning untuk XGBoost...
XGBoost Best Params: {'learning_rate': 0.026311893099709476, 'max_depth': 4, 'subsample': 0.6261547227081006, 'colsample_bytree': 0.6024835233755745, 'n_estimators': 800, 'random_state': 42}


### 5.3 K-Fold Triple Ensemble Prediction
Mengeksekusi siklus *Cross-Validation* dan mengekstraksi nilai probabilitas dari ketiga algoritma untuk digabungkan menjadi tebakan mutlak terhadap masa depan (Test Set).

In [12]:
test_preds_lgb = np.zeros(len(X_test_full))
test_preds_xgb = np.zeros(len(X_test_full))
test_preds_cat = np.zeros(len(X_test_full))

cat_params = {
    'iterations': 800,
    'learning_rate': 0.03,
    'depth': 8,
    'random_seed': 42,
    'verbose': False
}

cv_rmse_scores = []

print("\nMemulai Pelatihan K-Fold Ensemble...")
for fold, (train_index, val_index) in enumerate(tscv.split(X_train_full)):
    X_tr, X_va = X_train_full.iloc[train_index], X_train_full.iloc[val_index]
    y_tr, y_va = y_train_full.iloc[train_index], y_train_full.iloc[val_index]
    
    m_lgb = lgb.LGBMRegressor(**best_lgb_params)
    m_xgb = xgb.XGBRegressor(**best_xgb_params)
    m_cat = CatBoostRegressor(**cat_params)
    
    m_lgb.fit(X_tr, y_tr)
    m_xgb.fit(X_tr, y_tr, verbose=False)
    m_cat.fit(X_tr, y_tr)
    
    p_lgb = m_lgb.predict(X_va)
    p_xgb = m_xgb.predict(X_va)
    p_cat = m_cat.predict(X_va)
    
    blend = (p_lgb * 0.35) + (p_xgb * 0.35) + (p_cat * 0.30)
    fold_rmse = np.sqrt(mean_squared_error(y_va, blend))
    cv_rmse_scores.append(fold_rmse)
    print(f"Fold {fold+1} RMSE: {fold_rmse:.4f}")
    
    test_preds_lgb += m_lgb.predict(X_test_full) / tscv.n_splits
    test_preds_xgb += m_xgb.predict(X_test_full) / tscv.n_splits
    test_preds_cat += m_cat.predict(X_test_full) / tscv.n_splits

print(f"\nRata-rata K-Fold CV RMSE (Triple Blend): {np.mean(cv_rmse_scores):.4f}")


Memulai Pelatihan K-Fold Ensemble...
Fold 1 RMSE: 5.3585
Fold 2 RMSE: 2.2466
Fold 3 RMSE: 2.0008
Fold 4 RMSE: 5.2208
Fold 5 RMSE: 1.1773

Rata-rata K-Fold CV RMSE (Triple Blend): 3.2008


## 6. Output Final
Proyeksi ansambel disejajarkan dengan identitas waktu masa depan dan dikemas rapi.

In [13]:
final_blend = (test_preds_lgb * 0.35) + (test_preds_xgb * 0.35) + (test_preds_cat * 0.30)

test_data['tma_mdpl'] = final_blend
submission = test_data[['id', 'tma_mdpl']]

if not os.path.exists('../submissions'):
    os.makedirs('../submissions')

submission.to_csv('../submissions/submission.csv', index=False)
print("Seluruh komputasi tuntas. File submission.csv siap untuk medan pertempuran Kaggle.")

Seluruh komputasi tuntas. File submission.csv siap untuk medan pertempuran Kaggle.
